# Carga · Golden

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window
from datetime import datetime, timezone
import csv
import re
DEFAULTS = {'storageName': 'adlsmardata1412jmvcc', 'catalogo': 'fi_videojuegos', 'storageCredential': 'credential', 'container_metastore': 'metastore', 'carpeta_proyecto': 'fi_videojuegos_jmcc', 'raw_subdir': '', 'aplicar_grants': 'false'}
for clave, valor in DEFAULTS.items():
    dbutils.widgets.text(clave, valor)
PARAMS = {clave: dbutils.widgets.get(clave) for clave in DEFAULTS}
CATALOGO = PARAMS['catalogo'].strip()
STORAGE = PARAMS['storageName'].strip()
CARPETA_PROYECTO = PARAMS['carpeta_proyecto'].strip('/')
raw_subdir = PARAMS['raw_subdir'].strip('/')
assert re.fullmatch('[a-z][a-z0-9_]*', CATALOGO)
assert re.fullmatch('[a-z0-9]{3,24}', STORAGE)
assert re.fullmatch('[a-z0-9_/-]+', CARPETA_PROYECTO) and '..' not in CARPETA_PROYECTO
assert not raw_subdir or (re.fullmatch('[A-Za-z0-9_/-]+', raw_subdir) and '..' not in raw_subdir)
RUTA_RAW = f'abfss://raw@{STORAGE}.dfs.core.windows.net/' + raw_subdir
RUTA_RAW = RUTA_RAW.rstrip('/')
TABLAS_EXTERNAS = True

def q(s):
    return '`' + s.replace('`', '``') + '`'

def nombre(capa, tabla):
    assert capa in ('bronze', 'silver', 'golden')
    assert tabla.startswith('fi_'), 'Solo se permiten tablas de este proyecto.'
    return f'{q(CATALOGO)}.{q(capa)}.{q(tabla)}'

def ruta_delta(capa, tabla):
    return f'abfss://{capa}@{STORAGE}.dfs.core.windows.net/{CARPETA_PROYECTO}/{tabla}'

def preparar(df, capa, tabla):
    destino = nombre(capa, tabla)
    actual = spark.table(destino)
    assert [(f.name, f.dataType) for f in actual.schema] == [(f.name, f.dataType) for f in df.schema], f'Esquema diferente en {destino}. Revisar antes de reemplazar datos.'
    detalle = spark.sql(f'DESCRIBE DETAIL {destino}').first().asDict()
    assert detalle['format'].lower() == 'delta'
    if TABLAS_EXTERNAS:
        assert detalle['location'].rstrip('/') == ruta_delta(capa, tabla).rstrip('/'), f'Tabla ya registrada en otra ruta: {destino}. No se modifica.'
    else:
        tipo = spark.sql(f'DESCRIBE TABLE EXTENDED {destino}').collect()
        tipos = [r['data_type'] for r in tipo if r['col_name'].strip() == 'Type']
        assert tipos and tipos[0].upper() == 'MANAGED', f'{destino} ya es externa.'
    return destino

def guardar(df, capa, tabla):
    destino = preparar(df, capa, tabla)
    df.select(*spark.table(destino).columns).write.mode('overwrite').insertInto(destino)
    resultado = spark.table(destino)
    assert resultado.count() == df.count(), f'Conteo diferente: {destino}'
    print('OK', destino, resultado.count(), 'filas')
    return resultado

## G02 — Leer Silver
No vuelve a leer CSV ni realiza de nuevo los cruces.

In [0]:
silver = spark.table(nombre("silver", "fi_sl_ventas_enriquecidas"))
assert silver.count() == 16598
assert silver.select("rank_venta").distinct().count() == silver.count()

## G03 — Tabla de hechos
Una fila por Rank original; conserva también las ventas sin coincidencia.

In [0]:
hechos = (silver.withColumn("decada", (F.floor(F.col("anio") / 10) * 10).cast("int"))
          .withColumn("con_metadatos", F.col("hash_origen").isNotNull()))
hechos = guardar(hechos, "golden", "fi_gd_hechos_ventas")
base_valida = hechos.where("venta_valida")

## G04 — Ventas por plataforma y año
Sumas de millones de unidades estimadas. Las puntuaciones IGDB y Metacritic se promedian por separado.

In [0]:
plataformas = base_valida.groupBy("familia","plataforma","anio").agg(
    F.count("*").alias("registros_ventas"),
    F.sum("ventas_globales").alias("ventas_globales_millones"),
    F.sum("ventas_na").alias("ventas_na_millones"), F.sum("ventas_eu").alias("ventas_eu_millones"),
    F.sum("ventas_jp").alias("ventas_jp_millones"), F.sum("ventas_otras").alias("ventas_otras_millones"),
    F.avg("score_igdb").alias("promedio_igdb_0_100"), F.count("score_igdb").alias("n_score_igdb"),
    F.avg("meta_score").alias("promedio_meta_0_100"), F.count("meta_score").alias("n_meta_score"))
# Resultado intermedio: se muestra sin crear otra tabla.


## G05 — Ventas por género y familia
Usa género de VG Sales: no reparte una misma venta entre varios géneros de IGDB.

In [0]:
generos = base_valida.groupBy("familia","genero").agg(
    F.count("*").alias("registros_ventas"),
    F.sum("ventas_globales").alias("ventas_globales_millones"))
# Resultado intermedio: se muestra sin crear otra tabla.


## G06 — Cobertura y calidad
Cobertura calculada sobre TODAS las ventas; los promedios solo consideran datos disponibles. No son promedios de todos los juegos del mercado.

In [0]:
cobertura = hechos.groupBy("familia").agg(
    F.count("*").alias("ventas_fuente"),
    F.sum(F.col("con_metadatos").cast("int")).alias("ventas_con_metadatos"),
    F.sum(F.when(F.col("fi_num_candidatos") > 1,1).otherwise(0)).alias("coincidencias_ambiguas"),
    F.avg("score_igdb").alias("promedio_igdb_0_100"),
    F.avg("meta_score").alias("promedio_meta_0_100"), F.avg("user_score").alias("promedio_usuario_0_10"),
    F.avg("horas_principal").alias("promedio_horas_principal"))
cobertura = cobertura.withColumn("cobertura_pct", 100.0 * F.col("ventas_con_metadatos") / F.col("ventas_fuente"))
# Resultado intermedio: se muestra sin crear otra tabla.


## G07 — Conciliación final
Las sumas de Golden deben coincidir exactamente con Silver usando decimales, sin redondeo intermedio.

In [0]:
total = base_valida.agg(F.sum("ventas_globales")).first()[0]
assert total == plataformas.agg(F.sum("ventas_globales_millones")).first()[0]
assert total == generos.agg(F.sum("ventas_globales_millones")).first()[0]
assert hechos.count() == silver.count()
display(plataformas.orderBy(F.desc("ventas_globales_millones")).limit(20))
display(cobertura.orderBy("familia"))
print("Conciliacion de ventas validas, millones de unidades:", total)

## G08 — Sumar ventas por familia y año de lanzamiento



In [0]:
familias = ["Nintendo", "PlayStation", "Xbox"]

ventas_comparables = base_valida.filter(
    F.col("familia").isin(*familias)
)

print(
    "Registros excluidos por año desconocido:",
    ventas_comparables.filter(F.col("anio").isNull()).count()
)

ventas_familia_anio = (
    ventas_comparables
    .filter(F.col("anio").isNotNull())
    .groupBy("anio", "familia")
    .agg(
        F.count("*").alias("registros_juego_plataforma"),
        F.sum("ventas_globales").alias("ventas_globales_millones")
    )
)

display(ventas_familia_anio.orderBy("anio", "familia"))

## G09 — Ranking y participación de cada familia

In [0]:
from pyspark.sql.window import Window

por_anio = Window.partitionBy("anio")
orden_ventas = por_anio.orderBy(
    F.col("ventas_globales_millones").desc()
)

ranking = (
    ventas_familia_anio
    .withColumn(
        "puesto",
        F.dense_rank().over(orden_ventas)
    )
    .withColumn(
        "total_tres_familias_millones",
        F.sum("ventas_globales_millones").over(por_anio)
    )
    .withColumn(
        "participacion_pct",
        F.when(
            F.col("total_tres_familias_millones") > 0,
            F.round(
                100 * F.col("ventas_globales_millones")
                / F.col("total_tres_familias_millones"),
                2
            )
        )
    )
)

ranking = guardar(
    ranking,
    "golden",
    "fi_gd_ranking_familia_anio"
)

display(ranking.orderBy("anio", "puesto", "familia"))

## G10 — Mostrar quién lidera cada año


In [0]:
lideres = ranking.filter(F.col("puesto") == 1)

# Resultado intermedio: se muestra sin crear otra tabla.

display(
    lideres.select(
        "anio",
        "familia",
        "ventas_globales_millones",
        "participacion_pct"
    ).orderBy("anio", "familia")
)

## G11 — Mostrar lo mas vendido por familia


In [0]:
# G12 — Mejor juego por familia y década
from pyspark.sql import functions as F
from pyspark.sql.window import Window

ventas = (
    spark.table(nombre("golden", "fi_gd_hechos_ventas"))
    .filter(
        F.col("venta_valida")
        & F.col("familia").isin("Nintendo", "PlayStation", "Xbox")
        & F.col("anio").isNotNull()
    )
    .withColumn(
        "inicio_decada",
        (F.floor(F.col("anio") / 10) * 10).cast("int")
    )
    .withColumn(
        "decada",
        F.concat(
            F.col("inicio_decada").cast("string"),
            F.lit("–"),
            (F.col("inicio_decada") + 9).cast("string")
        )
    )
)

orden = Window.partitionBy("inicio_decada", "familia").orderBy(
    F.col("ventas_globales").desc(),
    F.col("rank_venta").asc()
)

clasificados = ventas.withColumn(
    "puesto", F.row_number().over(orden)
)

primero = (
    clasificados.filter(F.col("puesto") == 1)
    .select(
        "inicio_decada",
        "decada",
        "familia",
        F.col("titulo").alias("mejor_juego"),
        "plataforma",
        F.col("ventas_globales").alias("ventas_millones")
    )
)

segundo = (
    clasificados.filter(F.col("puesto") == 2)
    .select(
        "inicio_decada",
        "familia",
        F.col("titulo").alias("segundo_juego"),
        F.col("plataforma").alias("plataforma_segundo"),
        F.col("ventas_globales").alias("ventas_segundo_millones")
    )
)

resultado = (
    primero
    .join(segundo, ["inicio_decada", "familia"], "left")
    .withColumn(
        "supero_al_segundo_pct",
        F.when(
            F.col("ventas_segundo_millones") > 0,
            F.round(
                100.0
                * (
                    F.col("ventas_millones")
                    - F.col("ventas_segundo_millones")
                )
                / F.col("ventas_segundo_millones"),
                2
            )
        )
    )
)

resultado = guardar(
    resultado,
    "golden",
    "fi_gd_mejor_juego_familia_decada"
)

display(
    resultado
    .orderBy("inicio_decada", F.desc("ventas_millones"))
    .drop("inicio_decada")
)

In [0]:
assert hechos.count() == silver.count() == 16598
assert hechos.agg(F.sum('ventas_globales')).first()[0] == silver.agg(F.sum('ventas_globales')).first()[0]
assert cobertura.agg(F.sum('ventas_fuente')).first()[0] == hechos.count()
assert ranking.groupBy('anio','familia').count().where('count > 1').count() == 0
assert resultado.groupBy('inicio_decada','familia').count().where('count > 1').count() == 0
print('GOLDEN COMPLETADO: hechos, ranking anual y mejor juego por decada.')
print('Los años corresponden al lanzamiento; ventas en millones de unidades estimadas.')